# Data Description

- Background: 本資料為臺北市政府資料大平臺發布的「臺北市死傷交通事故資料」114 年明細，由臺北市政府警察局交通大隊提供。本組擬用於檢視夜間、無道路照明情況下的事故地點；死亡事故件數為主要篩選資訊之一，嚴重事故占比與事故總數為輔助資訊。
- date received: 20260929
- Path to data file: [臺北市資料大平臺－臺北市死傷交通事故資料](https://data.taipei/dataset/detail?id=2f238b4f-1b27-4085-93e9-d684ef0e2735)
- Codebook: [事故對照表.pdf](事故對照表.pdf)（112.07.01 後適用道路交通事故調查報告表，共 4 頁）
- Codebook SHA-256: `f40ded1ded1c816a38f421bc972cdc1320bbd280e647792207ec037d05fd3670`
- Unit of observation: 待確認。檔案每列皆為 `當事人序號 = 1`，並同時包含事故層級死傷人數與第一當事者欄位；不能只靠欄名確認每列是否等於一件事故。
- Sample period: 民國 114 年（2025 年）1 月至 12 月。
- Known issues:
    - 官方資料頁與 112 年 7 月 1 日後適用的調查表提供代碼定義，但本檔實際欄位比資料頁列出的欄位少，未含 `車輛撞擊部位2` 與 `肇因碼-主要`。
    - `光線` 實際值為 5、6、7 或缺失。官方代碼表將此欄標為「道路照明設備」：5＝有照明且開啟、6＝有照明未開啟或故障、7＝無照明；這不能單獨定義「夜間」，夜間時段規則仍待本組確認。
    - 官方 `受傷程度` 代碼只區分 24 小時內死亡、受傷、未受傷、不明、2–30 日內死亡，沒有「重傷」類別；目前無法由本檔可靠辨識重傷。
    - 沒有事故唯一識別碼；年月日、時分、行政區與地點組合仍有重複，不能直接當作已驗證的事故鍵。
    - 沒有車流量，後續結果不可稱為事故率；本資料也不能單獨識別照明改善的因果效果。
    - 原始資料含少量缺失值，詳見下方欄位完整性與摘要。
- Definition for each variable: 下方變數定義表依檔案欄名與本地 [事故對照表.pdf](事故對照表.pdf) 整理；代碼欄位的完整值標籤以該表為準。
- Data path: `data/raw/114年-臺北市死傷交通事故明細.csv`
- Role: `raw`
- Verification status: 2026-09-29 已核對檔案存在、UTF-8 with BOM 編碼、SHA-256、47 欄、22,762 列、樣本月份、欄位完整性、完整列重複、候選事故鍵與主要代碼分布；另已讀取本地 4 頁事故對照表並核對其 SHA-256。觀察單位、事故唯一鍵、夜間定義，以及重傷所需的其他資料來源或辨識方式仍待確認。
- SHA-256: `61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c`
- Next processing notebook: pending until created

本 Notebook 僅描述未修改的原始檔，不會清理或覆寫 CSV。

In [1]:
from pathlib import Path
import hashlib
import pandas as pd
from IPython.display import display


def find_project_root(start=None):
    start = Path.cwd() if start is None else Path(start)
    for candidate in (start.resolve(), *start.resolve().parents):
        if (candidate / 'AGENTS.md').is_file() and (candidate / 'templates').is_dir():
            return candidate
    raise FileNotFoundError('Open this notebook within the student project containing AGENTS.md and templates/.')


PROJECT_ROOT = find_project_root()


def project_file(relative_path, area, must_exist=True):
    rel = Path(relative_path)
    if rel.is_absolute() or '..' in rel.parts or '<' in str(rel):
        raise ValueError(f'Replace the placeholder with an exact project-relative path: {relative_path}')
    path = (PROJECT_ROOT / rel).resolve()
    allowed = (PROJECT_ROOT / 'data' / area).resolve()
    if not path.is_relative_to(allowed):
        raise ValueError(f'Expected a file inside data/{area}/: {relative_path}')
    if must_exist and not path.is_file():
        raise FileNotFoundError(path)
    return path


def sha256_file(path):
    digest = hashlib.sha256()
    with path.open('rb') as stream:
        for block in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(block)
    return digest.hexdigest()


def verify_input_sha256(path, expected):
    actual = sha256_file(path)
    if actual != expected.lower():
        raise ValueError(f'Input changed: identify the new version before loading {path.name}.')
    return actual

## Change for Your Own Data

以下程式綁定精確的專案相對路徑與已核對 SHA-256。所有欄位先以字串讀入，避免月份、日期與代碼的前導零消失；只有摘要所需欄位另行轉成數值，不回寫原始檔。

In [2]:
INPUT_RELATIVE_PATH = 'data/raw/114年-臺北市死傷交通事故明細.csv'
EXPECTED_SHA256 = '61f0bdb52a49176746ab8b66926224b294498c9e26b2dec6eefe44906fdffa0c'
input_data_file = project_file(INPUT_RELATIVE_PATH, 'raw')

## Summary

In [3]:
input_fingerprint = verify_input_sha256(input_data_file, EXPECTED_SHA256)
df = pd.read_csv(input_data_file, encoding='utf-8-sig', dtype='string')
overview = pd.Series({
    'data_path': INPUT_RELATIVE_PATH,
    'rows': len(df),
    'columns': len(df.columns),
    'exact_duplicate_rows': int(df.duplicated().sum()),
    'sample_year_values': ', '.join(df['發生年度'].dropna().unique()),
    'sample_month_values': ', '.join(sorted(df['發生月'].dropna().unique())),
    'sha256': input_fingerprint,
}, name='value').to_frame()
display(overview)

,value
data_path,data/raw/114年-臺北市死傷交通事故明細.csv
rows,22762
columns,47
exact_duplicate_rows,0
sample_year_values,114
sample_month_values,"01, 02, 03, 04, 05, 06, 07, 08, 09, 10, 11, 12"
sha256,61f0bdb52a49176746ab8b66926224b294498c9e26b2de...


### Sample

In [4]:
display(df.head(10))

,發生年度,發生月,發生日,發生時-Hours,發生分,區序,肇事地點,死亡人數,2-30日死亡人數,受傷人數,...,當事者行動狀態,駕駛資格情形,駕駛執照種類,飲酒情形,車輛撞擊部位1,肇因碼-個別,個人肇逃否,職業,座標-X,座標-Y
0,114,01,01,00,18,03中山區,中山區市民大道2段47號,0,0,1,...,9,1,10,1,11,16,1,21,121.5290038,25.04660303
1,114,01,01,00,31,10文山區,文山區新光路1段109號旁,0,0,3,...,1,1,10,1,14,19,1,21,121.5736892,24.9912752
2,114,01,01,00,35,03中山區,中山區北安路與大直街口,0,0,1,...,5,3,18,1,12,96,1,21,121.5471963,25.07984181
3,114,01,01,00,41,03中山區,中山區民族東路2號,0,0,1,...,1,1,4,1,8,19,1,21,121.5233047,25.0683953
4,114,01,01,00,57,11南港區,南港區忠孝東路6段與玉成街口,0,0,1,...,9,1,8,1,5,108,1,22,121.5832269,25.0458717
5,114,01,01,01,34,07信義區,信義區吳興街269巷18弄與吳興街281巷47弄口,0,0,3,...,15,1,10,1,11,74,1,21,121.5652003,25.0269334
6,114,01,01,02,15,07信義區,信義區基隆路2段107號之2,0,0,1,...,9,1,10,1,11,118,1,22,121.5575723,25.029541
7,114,01,01,06,26,10文山區,文山區指南路3段(岐山幹42),0,0,1,...,9,1,10,1,11,108,1,21,121.5972232,24.9724278
8,114,01,01,06,29,10文山區,文山區辛亥路4段與辛亥路4段101巷口,0,0,1,...,7,1,4,2,2,100,1,21,121.5583969,25.0064466
9,114,01,01,07,11,06松山區,松山區寶清街93號,0,0,2,...,1,1,8,1,4,19,1,21,121.5679891,25.0524534


### Definition for each variable

定義依 [臺北市資料大平臺](https://data.taipei/dataset/detail?id=2f238b4f-1b27-4085-93e9-d684ef0e2735) 與本地 [事故對照表.pdf](事故對照表.pdf) 整理；本地檔為 112 年 7 月 1 日後適用的道路交通事故調查報告表。

In [5]:
definitions = {
    '發生年度': '事故發生民國年度。', '發生月': '事故發生月份。', '發生日': '事故發生日。',
    '發生時-Hours': '事故發生小時（24 小時制）。', '發生分': '事故發生分鐘。',
    '區序': '臺北市行政區代碼與名稱。', '肇事地點': '事故發生地點文字。',
    '死亡人數': '事故死亡人數；與 2–30 日死亡人數的關係仍須依來源確認。',
    '2-30日死亡人數': '事故後 2–30 日內死亡人數。', '受傷人數': '事故受傷人數。',
    '當事人序號': '道路交通事故當事者順位；本檔觀察值全為 1。',
    '車種': '第一當事者車種／當事者區分類別代碼。', '天候': '事故現場天候代碼。',
    '光線': '道路照明設備代碼；本檔值 5＝有照明且開啟、6＝有照明未開啟或故障、7＝無照明。',
    '道路類別': '第一當事者所在道路類別代碼。', '速限-速度限制': '道路速限。',
    '道路型態': '道路型態代碼。', '事故位置': '事故位置代碼。',
    '路面狀況1': '路面鋪裝代碼。', '路面狀況2': '路面狀態代碼。', '路面狀況3': '路面缺陷代碼。',
    '道路障礙1': '道路障礙物代碼。', '道路障礙2': '視距／遮蔽物狀況代碼。',
    '號誌1': '號誌種類代碼。', '號誌2': '號誌動作代碼。',
    '車道劃分-分向': '車道分向設施代碼。', '車道劃分-分道1': '快車道或一般車道間分道設施代碼。',
    '車道劃分-分道2': '快慢車道間分道設施代碼。', '車道劃分-分道3': '路面邊線代碼。',
    '事故類型及型態': '人與車、車與車、車輛本身或平交道等事故型態代碼。',
    '性別': '第一當事者性別／屬性別代碼。', '年齡': '第一當事者事故發生時年齡。',
    '受傷程度': '第一當事者傷亡狀態代碼：1＝24 小時內死亡、2＝受傷、3＝未受傷、4＝不明、5＝2–30 日內死亡。',
    '主要傷處': '第一當事者主要傷處代碼。', '保護裝置': '第一當事者保護裝備使用代碼。',
    '行動電話': '行動電話、電腦或其他相類功能裝置使用情形代碼。',
    '車輛用途': '第一當事者車輛用途代碼。', '當事者行動狀態': '第一當事者事故時行動狀態代碼。',
    '駕駛資格情形': '第一當事者駕照狀態代碼。', '駕駛執照種類': '第一當事者駕照種類代碼。',
    '飲酒情形': '第一當事者飲酒或酒精檢測情形代碼。', '車輛撞擊部位1': '第一當事者最初車輛撞擊部位代碼。',
    '肇因碼-個別': '第一當事者個別肇因初步研判代碼。', '個人肇逃否': '第一當事者是否肇事逃逸代碼。',
    '職業': '第一當事者職業代碼。', '座標-X': '事故位置經度。', '座標-Y': '事故位置緯度。'
}
variable_definitions = pd.DataFrame({
    'variable': df.columns,
    'definition': [definitions.get(column, '待確認') for column in df.columns],
})
display(variable_definitions)
assert set(df.columns) == set(definitions), '變數定義未完整涵蓋實際欄位。'

,variable,definition
0,發生年度,事故發生民國年度。
1,發生月,事故發生月份。
2,發生日,事故發生日。
3,發生時-Hours,事故發生小時（24 小時制）。
4,發生分,事故發生分鐘。
5,區序,臺北市行政區代碼與名稱。
6,肇事地點,事故發生地點文字。
7,死亡人數,事故死亡人數；與 2–30 日死亡人數的關係仍須依來源確認。
8,2-30日死亡人數,事故後 2–30 日內死亡人數。
9,受傷人數,事故受傷人數。


### Summary Stats

In [6]:
field_completeness = pd.DataFrame({
    'dtype_on_read': df.dtypes.astype(str),
    'non_missing_n': df.notna().sum(),
    'missing_n': df.isna().sum(),
    'unique_non_missing': df.nunique(dropna=True),
}).sort_values(['missing_n', 'unique_non_missing'], ascending=[False, True])
display(field_completeness)

,dtype_on_read,non_missing_n,missing_n,unique_non_missing
行動電話,string,22543,219,5
保護裝置,string,22543,219,6
受傷程度,string,22547,215,5
主要傷處,string,22547,215,11
職業,string,22551,211,12
駕駛執照種類,string,22553,209,19
車輛撞擊部位1,string,22557,205,17
車輛用途,string,22562,200,5
當事者行動狀態,string,22563,199,21
駕駛資格情形,string,22564,198,7


In [7]:
numeric_columns = ['死亡人數', '2-30日死亡人數', '受傷人數', '速限-速度限制', '年齡', '座標-X', '座標-Y']
numeric_df = df[numeric_columns].apply(pd.to_numeric, errors='coerce')
numeric_summary = numeric_df.agg(['mean', 'min', lambda x: x.quantile(.25), 'median', lambda x: x.quantile(.75), 'max', 'std', 'count']).T
numeric_summary.columns = ['mean', 'min', 'q25', 'q50', 'q75', 'max', 'sd', 'N']
display(numeric_summary)

,mean,min,q25,q50,q75,max,sd,N
死亡人數,0.002549,0.00000,0.000000,0.000000,0.000000,2.000000,0.051287,22755.0
2-30日死亡人數,0.000132,0.00000,0.000000,0.000000,0.000000,1.000000,0.011482,22752.0
受傷人數,1.326875,0.00000,1.000000,1.000000,2.000000,21.000000,0.597007,22761.0
速限-速度限制,44.145971,0.00000,40.000000,50.000000,50.000000,80.000000,10.055842,22758.0
年齡,44.376747,0.00000,30.000000,44.000000,58.000000,113.000000,17.240335,22750.0
座標-X,121.540491,121.46118,121.516802,121.536655,121.562316,121.652018,0.030742,22760.0
座標-Y,25.054330,24.96593,25.034563,25.049958,25.068917,25.183295,0.032454,22760.0


In [8]:
categorical_columns = ['發生月', '區序', '當事人序號', '車種', '天候', '光線', '道路類別', '道路型態', '事故位置', '事故類型及型態', '性別', '受傷程度']
top_rows = []
for column in categorical_columns:
    counts = df[column].value_counts(dropna=False).head(10)
    valid_n = int(df[column].notna().sum())
    for value, count in counts.items():
        top_rows.append({
            'variable': column, 'value': '<missing>' if pd.isna(value) else value,
            'count': int(count),
            'percent_of_valid': (100 * count / valid_n) if valid_n and not pd.isna(value) else pd.NA,
            'valid_N': valid_n,
        })
categorical_summary = pd.DataFrame(top_rows)
display(categorical_summary)

,variable,value,count,percent_of_valid,valid_N
0,發生月,12,2063,9.063351,22762
1,發生月,03,2020,8.87444,22762
2,發生月,11,1967,8.641596,22762
3,發生月,10,1951,8.571303,22762
4,發生月,08,1949,8.562516,22762
...,...,...,...,...,...
84,受傷程度,2,10281,45.598084,22547
85,受傷程度,4,435,1.929303,22547
86,受傷程度,<missing>,215,<NA>,22547
87,受傷程度,1,28,0.124185,22547


### Potential key and research-variable checks

下列檢查只呈現資料結構，不把候選鍵視為正式事故識別碼，也不先建立夜間或重傷變數。

In [9]:
candidate_key = ['發生年度', '發生月', '發生日', '發生時-Hours', '發生分', '區序', '肇事地點']
candidate_sizes = df.groupby(candidate_key, dropna=False).size()
key_checks = pd.Series({
    'candidate_groups': int(candidate_sizes.size),
    'candidate_groups_with_multiple_rows': int((candidate_sizes > 1).sum()),
    'maximum_rows_in_candidate_group': int(candidate_sizes.max()),
    'all_party_sequence_equal_1': bool(df['當事人序號'].dropna().eq('1').all()),
    'rows_missing_coordinates': int(df[['座標-X', '座標-Y']].isna().any(axis=1).sum()),
}, name='value').to_frame()
display(key_checks)
display(pd.DataFrame({
    '光線_count': df['光線'].value_counts(dropna=False),
    '光線_percent_all_rows': df['光線'].value_counts(dropna=False, normalize=True).mul(100),
}))
display(pd.DataFrame({
    '受傷程度_count': df['受傷程度'].value_counts(dropna=False),
    '受傷程度_percent_all_rows': df['受傷程度'].value_counts(dropna=False, normalize=True).mul(100),
}))

,value
candidate_groups,22751
candidate_groups_with_multiple_rows,11
maximum_rows_in_candidate_group,2
all_party_sequence_equal_1,True
rows_missing_coordinates,2


,光線_count,光線_percent_all_rows
6,9296,40.839996
5,9058,39.794394
7,4396,19.31289
<NA>,12,0.052719


,受傷程度_count,受傷程度_percent_all_rows
3,11798,51.832001
2,10281,45.167384
4,435,1.91108
<NA>,215,0.944557
1,28,0.123012
5,5,0.021966


### Interpretation and next checks

- 本檔可直接描述 2025 年臺北市死傷交通事故明細中的事故時間、地點、死傷人數、道路照明設備代碼與座標。
- 目前不能可靠建立「死亡或重傷事故」：死亡可由死亡人數欄位描述，但官方受傷程度沒有重傷分類。
- `光線 = 7` 依官方表為無照明，但仍須另定義夜間時段；不能把無照明直接等同夜間。
- 候選事故鍵有重複，且沒有官方事故 ID；在彙整到地點或路口前，須先確認每列觀察單位及事故去重規則。
- `肇事地點` 是自由文字，座標也有缺失；路口層級標準化與空間配對需要另建資料處理 Notebook 並驗證。
- 本資料沒有車流量，因此後續呈現應使用事故件數或事故占比，而非事故率；任何地點差異都不能單憑本資料解釋為照明改善的因果效果。